# Entrenamiento de segmentación

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 albumentations==2.0.8

## Imports

In [ ]:
from pathlib import Path

import albumentations as A
import cv2
from IPython.display import Markdown, display
from ultralytics import YOLO

## Rutas y parámetros

In [ ]:
# Carpeta que contiene datasets/: la del notebook en el servidor o la raíz del repo
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
DATA = REPO / "datasets/floor-plan-cad-yolo/floor-plan-cad-yolo.yaml"
RUNS = REPO / "runs"

MODEL = "yolo26m-seg.pt"
IMGSZ = 1024
BATCH = 2
EPOCHS = 120
SEED = 42
DEVICE = 0
PATIENCE = 20
RESUME = False  # True retoma la corrida cortada desde su last.pt
NAME = Path(MODEL).stem
RUN_DIR = RUNS / NAME  # el entrenamiento la actualiza si Ultralytics usa otro nombre (p. ej. yolo26m-seg2)

assert DATA.exists(), f"No existe el dataset: {DATA}"

## Augmentations

> Simulan planos analógicos escaneados.

In [ ]:
class StrokeMorphology(A.ImageOnlyTransform):
    """Erosión o dilatación de la imagen sola.

    A.Morphological también modifica las cajas: Ultralytics se las pasa y
    OpenCV falla con imágenes de más de 512 px de ancho.
    """

    def __init__(self, scale=2, operation="erosion", p=0.5):
        super().__init__(p=p)
        self.scale = scale
        self.operation = operation

    def apply(self, img, **params):
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (self.scale, self.scale))
        return cv2.erode(img, kernel) if self.operation == "erosion" else cv2.dilate(img, kernel)

    def get_transform_init_args_names(self):
        return ("scale", "operation")


# Degradaciones de escaneo; Ultralytics las aplica solo en entrenamiento
AUGMENTATIONS = [
    A.RandomBrightnessContrast(p=0.5),                         # tono del papel
    A.RandomGamma(p=0.3),
    A.Illumination(p=0.3),                                     # luz despareja
    A.Spatter(mode="mud", p=0.3),                              # manchas
    A.OneOf([StrokeMorphology(scale=2, operation="erosion"),   # trazo más grueso
             StrokeMorphology(scale=2, operation="dilation")], # trazo más fino
            p=0.3),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.3),                # ruido de escaneo
    A.SaltAndPepper(amount=(0.001, 0.01), p=0.2),              # motas
    A.Defocus(radius=(1, 3), p=0.2),                           # desenfoque
    A.ImageCompression(quality_range=(40, 90), p=0.3),         # artefactos JPG
    A.ToGray(p=1.0),                                           # BLD-AR está en escala de grises
]

## Entrenamiento

In [ ]:
if RESUME:
    # Usa los mismos argumentos y augmentations guardados en el checkpoint
    model = YOLO(RUN_DIR / "weights" / "last.pt")
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(DATA), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS,
        seed=SEED, deterministic=True, device=DEVICE, patience=PATIENCE,
        project=str(RUNS), name=NAME,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,  # planos en gris: sin tinte, solo brillo
        degrees=3.0,                      # escaneo levemente torcido
        flipud=0.5, fliplr=0.5,           # el plano no tiene arriba ni abajo
        mosaic=1.0, scale=0.5,
        augmentations=AUGMENTATIONS,
    )
RUN_DIR = Path(model.trainer.save_dir)
print(RUN_DIR)

## Evaluación en test

In [ ]:
best = YOLO(RUN_DIR / "weights" / "best.pt")
metrics = best.val(data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                   project=str(RUN_DIR), name="test")

## Métricas de test

In [ ]:
rows = [("Box", metrics.box.map50, metrics.box.map),
        ("Seg", metrics.seg.map50, metrics.seg.map)]
display(Markdown(
    f"**Test · {MODEL} · imgsz={IMGSZ}**\n\n| | mAP50 | mAP50-95 |\n|---|---|---|\n"
    + "\n".join(f"| {name} | {m50:.3f} | {m:.3f} |" for name, m50, m in rows)))